# 패키지 설치 및 데이터 로딩

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ✅ 한글 폰트 설정 (Mac용)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 1) 엑셀 읽기 (프로젝트 루트의 data/Engagement_interaction.xlsx)
df = pd.read_excel(os.path.join('Online Retail.xlsx'))

In [54]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


# 데이터 통계 보기

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [57]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [58]:
# 송장번호별로 고유한 CustomerID 개수 계산
invoice_customer_counts = df.groupby('InvoiceNo')['CustomerID'].nunique()

# 고객 아이디가 2개 이상인 송장번호만 필터링
multi_customer_invoices = invoice_customer_counts[invoice_customer_counts > 1]

if not multi_customer_invoices.empty:
    print(f"아이디가 여러 개인 송장 수: {len(multi_customer_invoices)}개")
    print("-" * 30)
    print(multi_customer_invoices)
else:
    print("모든 송장 번호에 고객 아이디가 하나씩만 매칭되어 있습니다.")

모든 송장 번호에 고객 아이디가 하나씩만 매칭되어 있습니다.


In [59]:
# CustomerID별로 고유한 InvoiceNo의 개수를 계산
reorder_check = df.groupby('CustomerID')['InvoiceNo'].nunique()

# 송장 번호가 2개 이상인 (재구매) 고객만 필터링
returning_customers = reorder_check[reorder_check > 1]

if not returning_customers.empty:
    print(f"재구매 고객 수: {len(returning_customers)}명")
    print(f"재구매 고객들의 총 송장 수: {returning_customers.sum()}개")
    print("-" * 30)
    print("상위 재구매 고객(ID별 송장 건수):")
    print(returning_customers.sort_values(ascending=False).head(10))
else:
    print("모든 고객이 단 한 번씩만 구매했습니다 (재구매 없음).")

재구매 고객 수: 3059명
재구매 고객들의 총 송장 수: 20877개
------------------------------
상위 재구매 고객(ID별 송장 건수):
CustomerID
14911.0    248
12748.0    224
17841.0    169
14606.0    128
13089.0    118
15311.0    118
12971.0     89
14527.0     86
13408.0     81
14646.0     77
Name: InvoiceNo, dtype: int64


# 결측값 처리
- CustomerID 결측값 : 비회원처리해서 송장번호에 900000을 더해 회원과 구분
- Description 결측값 : 동일한 StockCode중 값이 있는 것들로 채워둠 그리고 나머지는 단가 0원(환불건x)이라 이상치로 간주하고 삭제 

In [61]:
# 1. InvoiceNo에서 숫자만 추출 (문자 'C' 등이 포함되어 있을 수 있으므로)
# 숫자가 아닌 문자는 제거하고 정수로 변환
df['temp_invoice_num'] = df['InvoiceNo'].astype(str).str.extract('(\d+)').astype(float)

# 2. CustomerID가 결측치인 경우에만 임시 ID 부여
# 기존 ID와 겹치지 않도록 아주 큰 값(예: 900,000)을 더해줍니다.
df['CustomerID_Final'] = df['CustomerID'].fillna(df['temp_invoice_num'] + 900000)

# 3. 불필요한 임시 컬럼 삭제 및 타입 통일
df = df.drop(columns=['temp_invoice_num'])
df['CustomerID_Final'] = df['CustomerID_Final'].astype(int) # 이제 결측치가 없으므로 int 변환 가능

print(df[['InvoiceNo', 'CustomerID', 'CustomerID_Final']].head())

  InvoiceNo  CustomerID  CustomerID_Final
0    536365     17850.0             17850
1    536365     17850.0             17850
2    536365     17850.0             17850
3    536365     17850.0             17850
4    536365     17850.0             17850


In [62]:
df.isnull().sum()

InvoiceNo                0
StockCode                0
Description           1454
Quantity                 0
InvoiceDate              0
UnitPrice                0
CustomerID          135080
Country                  0
CustomerID_Final         0
dtype: int64

In [63]:
df = df.drop(columns=['CustomerID'])

In [64]:
df['CustomerID'] = df['CustomerID_Final']

In [65]:
df = df.drop(columns=['CustomerID_Final'])

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   Country      541909 non-null  object        
 7   CustomerID   541909 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 33.1+ MB


In [67]:
# StockCode별로 Description의 첫 번째 값을 가져와서 매핑 딕셔너리 생성
stock_desc_map = df.dropna(subset=['Description']).groupby('StockCode')['Description'].first()

# 결측치를 매핑 정보로 채우기
df['Description'] = df['Description'].fillna(df['StockCode'].map(stock_desc_map))

# 남은 결측치 확인
print(f"남은 Description 결측치: {df['Description'].isnull().sum()}")

남은 Description 결측치: 112


In [81]:
# Description이 여전히 결측치인 행들만 추출
remaining_nulls = df[df['Description'].isnull()].copy()

# 1. 환불건(InvoiceNo가 C로 시작)인지 확인
remaining_nulls['is_refund'] = remaining_nulls['InvoiceNo'].astype(str).str.startswith('C')

# 2. 결과 요약 출력
print("--- 남은 결측치 112개 분석 ---")
print(f"환불건(C) 개수: {remaining_nulls['is_refund'].sum()}개")
print(f"정상주문 개수: {len(remaining_nulls) - remaining_nulls['is_refund'].sum()}개")

# 3. 단가(UnitPrice)가 0원인 비율 확인
zero_price_count = (remaining_nulls['UnitPrice'] == 0).sum()
print(f"단가가 0원인 행: {zero_price_count}개")

# 4. 실제 데이터 샘플 확인
display(remaining_nulls[['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice', 'is_refund']].head(10))

--- 남은 결측치 112개 분석 ---
환불건(C) 개수: 0개
정상주문 개수: 112개
단가가 0원인 행: 112개


,InvoiceNo,StockCode,Quantity,UnitPrice,is_refund
1970,536545,21134,1,0.0,False
1987,536549,85226A,1,0.0,False
1988,536550,85044,1,0.0,False
2024,536552,20950,1,0.0,False
2026,536554,84670,23,0.0,False
7187,536995,35951,57,0.0,False
7193,537001,21653,-6,0.0,False
19628,537875,20849,1,0.0,False
19631,537878,72803B,1,0.0,False
21782,538133,85018C,3,0.0,False


In [83]:
# Description이 결측치인 행만 선택해서 삭제
df = df.dropna(subset=['Description'])

# 확인: 이제 결측치가 0이어야 합니다.
print(f"삭제 후 Description 결측치: {df['Description'].isnull().sum()}")

삭제 후 Description 결측치: 0


# 이상치 처리
- 단가가 0이거나 음수 데이터 중에서 C(환불)데이터가 아닌 경우 : 행 삭제

In [85]:
# 1. 단가가 0원인 데이터가 또 있는지 확인
zero_price = df[df['UnitPrice'] == 0]
print(f"단가가 0원인 행 수: {len(zero_price)}")

# 2. 수량이 마이너스인 데이터가 또 있는지 확인 (반품/취소 가능성)
negative_quantity = df[df['Quantity'] < 0]
print(f"수량이 0보다 작은 행 수: {len(negative_quantity)}")

단가가 0원인 행 수: 2403
수량이 0보다 작은 행 수: 10527


In [48]:
# 수량이 음수이면서 InvoiceNo가 C로 시작하는 비율 확인
cancelled_orders = df[df['Quantity'] < 0]
is_c_start = cancelled_orders['InvoiceNo'].astype(str).str.startswith('C').mean() * 100
print(f"음수 수량 중 C로 시작하는 비율: {is_c_start:.2f}%")

음수 수량 중 C로 시작하는 비율: 88.23%


In [88]:
# 음수 수량 중 InvoiceNo가 'C'로 시작하지 않는 데이터만 추출
weird_negative = df[(df['Quantity'] < 0) & (~df['InvoiceNo'].astype(str).str.startswith('C'))]

print(f"정체가 불분명한 음수 데이터 수: {len(weird_negative)}개")

# 이 데이터들의 Description과 UnitPrice 확인
print(weird_negative['Description'].value_counts().head(10))
print(f"\n이 데이터들의 평균 단가: {weird_negative['UnitPrice'].mean()}")

정체가 불분명한 음수 데이터 수: 1239개
Description
check                     120
damages                    45
?                          42
damaged                    42
sold as set on dotcom      20
Damaged                    14
thrown away                 9
Unsaleable, destroyed.      9
??                          7
wet damaged                 5
Name: count, dtype: int64

이 데이터들의 평균 단가: 0.0


In [91]:
# [제거 조건 정의]
# 수량(Quantity)이 0보다 작으면서 동시에 단가(UnitPrice)가 0인 데이터가 우리가 찾은 노이즈입니다.
noise_condition = (df['Quantity'] < 0) & (df['UnitPrice'] == 0)

# ~ 기호를 사용해 해당 조건이 '아닌' 데이터들만 남깁니다.
df_final = df[~noise_condition].copy()

# 결과 확인
print(f"제거 전 데이터 수: {len(df)}")
print(f"제거 후 데이터 수: {len(df_final)}")
print(f"제거된 노이즈 수: {len(df) - len(df_final)} (우리가 찾던 1239개여야 합니다)")

# 반품(C) 데이터가 잘 남아있는지 최종 확인
print(f"남아있는 반품('C') 데이터 수: {df_final['InvoiceNo'].astype(str).str.startswith('C').sum()}개")

제거 전 데이터 수: 541797
제거 후 데이터 수: 540558
제거된 노이즈 수: 1239 (우리가 찾던 1239개여야 합니다)
남아있는 반품('C') 데이터 수: 9288개


# 파생변수 생성

## 카테고리 1 : 기본 마케팅 변수
선행연구(Chen et al., 2012)와 겹치나, 모델의 베이스라인 역할을 하므로 반드시 포함함.

| 변수명 | 계산 방법 | 활용 목적 |
| :--- | :--- | :--- |
| **Recency** | 마지막 구매일 ~ 기준일(2011-12-10) 일수 | Main 종속변수(이탈) 라벨링 기준 |
| **Frequency** | 고객별 고유 InvoiceNo 수 | 구매 충성도 측정  |
| **Monetary** | 고객별 총 Revenue 합계 | 고객 가치 측정  |
| **AOV** | Monetary / Frequency | 평균 객단가 수준 파악|
| **Avg Items per Order** | 평균 장바구니 아이템 수 | 탐색 vs 습관 구분  |
| **Active Months** | 구매가 발생한 월수 | 구매 지속성 측정  |
| **Purchase Span** | 첫 구매 ~ 마지막 구매 일수 | 고객 생애 길이 파악  |

In [103]:
# 기준일 설정 (제안서 기준)
reference_date = pd.to_datetime('2011-12-10')

# 총 매출(Revenue) 컬럼 미리 생성
df_final['Revenue'] = df_final['Quantity'] * df_final['UnitPrice']

In [107]:
import pandas as pd

# 1. 기준일 설정 (제안서 기준: 2011-12-10) 
reference_date = pd.to_datetime('2011-12-10')

# 2. 고객별 기본 집계 수행
rfm_features = df_final.groupby('CustomerID').agg({
    'InvoiceDate': [
        lambda x: (reference_date - x.max()).days,  # Recency 계산 
        lambda x: x.dt.to_period('M').nunique(),    # Active Months 계산 
        lambda x: (x.max() - x.min()).days          # Purchase Span 계산 
    ],
    'InvoiceNo': 'nunique',                         # Frequency 계산 
    'Revenue': 'sum'                                # Monetary 계산 
})

# 3. 멀티 인덱스 컬럼명 정리
rfm_features.columns = ['Recency', 'Active_Months', 'Purchase_Span', 'Frequency', 'Monetary']

# 4. 추가 파생변수 계산 (AOV & Avg Items per Order) 
rfm_features['AOV'] = rfm_features['Monetary'] / rfm_features['Frequency']

# 주문당 평균 아이템 수 계산 (총 수량 / 고유 주문 수) [cite: 45, 52]
total_quantity = df_final.groupby('CustomerID')['Quantity'].sum()
rfm_features['Avg_Items_per_Order'] = total_quantity / rfm_features['Frequency']

# 5. 인덱스 초기화 및 결과 확인
rfm_features = rfm_features.reset_index()
print(rfm_features.head())

   CustomerID  Recency  Active_Months  Purchase_Span  Frequency  Monetary  \
0       12346      325              1              0          2      0.00   
1       12347        2              7            365          7   4310.00   
2       12348       75              4            282          4   1797.24   
3       12349       18              1              0          1   1757.55   
4       12350      310              1              0          1    334.40   

           AOV  Avg_Items_per_Order  
0     0.000000             0.000000  
1   615.714286           351.142857  
2   449.310000           585.250000  
3  1757.550000           631.000000  
4   334.400000           197.000000  


In [109]:
# AOV (Average Order Value): 평균 객단가
rfm_features['AOV'] = rfm_features['Monetary'] / rfm_features['Frequency']

# Avg Items per Order: 주문당 평균 장바구니 아이템 수
# (전체 수량의 합 / 고유 주문 수)
total_quantity = df_final.groupby('CustomerID')['Quantity'].sum()
rfm_features['Avg_Items_per_Order'] = total_quantity / rfm_features['Frequency']

# 인덱스로 되어있는 CustomerID를 컬럼으로 변환
rfm_features = rfm_features.reset_index()

## 카테고리 2: 시간 패턴 변수 (타성 관련)
논문 초안의 '타성' 넛지 유형과 직결되는 변수군임.

| 변수명 | 계산 방법 | 넛지 연결 |
| :--- | :--- | :--- |
| **Hour Entropy** | 결제 시간대 분포의 Shannon 엔트로피 | 낮을수록 특정 시간에 고착된 타성 상태로 분류 |
| **DayOfWeek Entropy** | 요일 분포의 Shannon 엔트로피 | 낮을수록 특정 요일에 고착된 타성 상태로 분류 |
| **Preferred Hour** | 가장 많이 구매한 시간대 | 구매 앵커링 시점 파악 |
| **Weekend Ratio** | 전체 구매 중 주말 비율 | 라이프스타일 패턴 파악 |
| **Inter-Purchase Interval Mean** | 연속 구매 간격 평균(일) | 구매 주기 측정 |
| **Inter-Purchase Interval Std** | 구매 간격 표준편차 | 규칙성 vs 불규칙성 측정 |
| **CV of Purchase Interval** | 간격 Std / 간격 Mean | 구매 리듬 안정성 측정 |
| **Month Entropy** | 구매 월 분포의 Shannon 엔트로피 | 계절성 패턴 파악 |

In [115]:
from scipy.stats import entropy

# 1. 시간적 맥락 추출 (요일, 시간, 월) 
df_final['hour'] = df_final['InvoiceDate'].dt.hour
df_final['dayofweek'] = df_final['InvoiceDate'].dt.dayofweek
df_final['month'] = df_final['InvoiceDate'].dt.month
df_final['is_weekend'] = df_final['dayofweek'].isin([5, 6]).astype(int)

# 2. Shannon 엔트로피 계산 함수 정의 
def get_shannon_entropy(series):
    # 각 범주별 확률 분포 계산
    prob_dist = series.value_counts(normalize=True)
    return entropy(prob_dist)

# 3. 고객별 시간 패턴 집계 (Hour, Day, Month Entropy 및 Preferred Hour 등)
time_features = df_final.groupby('CustomerID').agg({
    'hour': [get_shannon_entropy, lambda x: x.mode()[0] if not x.mode().empty else np.nan],
    'dayofweek': get_shannon_entropy,
    'month': get_shannon_entropy,
    'is_weekend': 'mean'
})

# 컬럼명 정리
time_features.columns = ['Hour_Entropy', 'Preferred_Hour', 'DayOfWeek_Entropy', 'Month_Entropy', 'Weekend_Ratio']

# 4. 구매 간격(Inter-Purchase Interval) 관련 변수 생성
# 동일 주문(InvoiceNo) 내 중복 제거 후 고객별 구매 시점 정렬 
order_dates = df_final[['CustomerID', 'InvoiceNo', 'InvoiceDate']].drop_duplicates().sort_values(['CustomerID', 'InvoiceDate'])
order_dates['prev_date'] = order_dates.groupby('CustomerID')['InvoiceDate'].shift(1)

# 연속 구매 간격(일 단위) 계산
order_dates['interval'] = (order_dates['InvoiceDate'] - order_dates['prev_date']).dt.days

# 고객별 간격 통계 산출 (Mean, Std, CV) 
interval_stats = order_dates.groupby('CustomerID')['interval'].agg(['mean', 'std']).rename(columns={'mean': 'IPI_Mean', 'std': 'IPI_Std'})
interval_stats['CV_Interval'] = interval_stats['IPI_Std'] / interval_stats['IPI_Mean']

# 5. 기존 고객 단위 데이터(rfm_features)에 순차적으로 병합 
rfm_features = pd.merge(rfm_features, time_features, on='CustomerID', how='left')
rfm_features = pd.merge(rfm_features, interval_stats, on='CustomerID', how='left')

# 시계열이 짧아(3회 미만) 계산 불가한 변수는 결측치 처리하거나 0으로 대체 
rfm_features = rfm_features.fillna(0)

# 결과 확인
print(rfm_features.head())

   index  CustomerID  Recency  Active_Months  Purchase_Span  Frequency  \
0      0       12346      325              1              0          2   
1      1       12347        2              7            365          7   
2      2       12348       75              4            282          4   
3      3       12349       18              1              0          1   
4      4       12350      310              1              0          1   

   Monetary          AOV  Avg_Items_per_Order  Hour_Entropy  Preferred_Hour  \
0      0.00     0.000000                  0.0      0.000000              10   
1   4310.00   615.714286                  0.0      1.636439              14   
2   1797.24   449.310000                  0.0      0.923106              19   
3   1757.55  1757.550000                  0.0      0.000000               9   
4    334.40   334.400000                  0.0      0.000000              16   

   DayOfWeek_Entropy  Month_Entropy  Weekend_Ratio   IPI_Mean    IPI_Std  \
0   

## 카테고리 3: 장바구니 구성 변수 (인지적 효율성 관련)
사용자의 구매 규모와 품목 다양성을 통해 인지 자원 활용 패턴을 분석함.

| 변수명 | 계산 방법 | 넛지 연결 |
| :--- | :--- | :--- |
| **Basket Diversity Index** | 주문당 Unique Items / Total Quantity 평균 | 인지적 효율성 측정 |
| **Avg Basket Size** | 주문당 총 Quantity 평균 | 구매 규모 파악 |
| **Avg Unique Items per Basket** | 주문당 고유 품목 수 평균 | 탐색 vs 집중 구매 구분 |
| **Large Basket Ratio** | Quantity > 10인 주문 비율 | 도매·대량 구매 성향 파악 |
| **Single Item Order Ratio** | 단품 주문 비율 | 목적 구매 성향 파악 |
| **Max Single Order Revenue** | 단일 주문 최대 매출 | 충동·특별 구매 탐지 |

In [118]:
# 1. 주문(InvoiceNo) 단위로 기본 지표 계산
order_basket_stats = df_final.groupby(['CustomerID', 'InvoiceNo']).agg({
    'StockCode': 'nunique',      # 주문당 고유 품목 수
    'Quantity': 'sum',           # 주문당 총 수량
    'Revenue': 'sum'             # 주문당 총 매출
}).reset_index()

# 2. 주문 단위의 파생 지표 생성
# Basket Diversity Index: 고유 품목 수 / 총 수량
order_basket_stats['diversity_index'] = order_basket_stats['StockCode'] / order_basket_stats['Quantity']

# Large Basket 여부 (주문 총 수량이 10개 초과인 경우)
order_basket_stats['is_large_basket'] = (order_basket_stats['Quantity'] > 10).astype(int)

# 단품 주문 여부 (고유 품목 수가 1개인 경우)
order_basket_stats['is_single_item'] = (order_basket_stats['StockCode'] == 1).astype(int)

# 3. 고객(CustomerID) 단위로 최종 요약 집계
category_3_features = order_basket_stats.groupby('CustomerID').agg({
    'diversity_index': 'mean',          # Basket Diversity Index
    'Quantity': 'mean',                 # Avg Basket Size
    'StockCode': 'mean',                # Avg Unique Items per Basket
    'is_large_basket': 'mean',          # Large Basket Ratio
    'is_single_item': 'mean',           # Single Item Order Ratio
    'Revenue': 'max'                    # Max Single Order Revenue
}).rename(columns={
    'diversity_index': 'Basket_Diversity_Index',
    'Quantity': 'Avg_Basket_Size',
    'StockCode': 'Avg_Unique_Items_per_Basket',
    'is_large_basket': 'Large_Basket_Ratio',
    'is_single_item': 'Single_Item_Order_Ratio',
    'Revenue': 'Max_Single_Order_Revenue'
})

# 4. 기존 고객 단위 데이터(rfm_features)에 병합
rfm_features = pd.merge(rfm_features, category_3_features, on='CustomerID', how='left')

# 결측치 처리
rfm_features = rfm_features.fillna(0)

# 결과 확인
print(rfm_features[['CustomerID', 'Basket_Diversity_Index', 'Avg_Basket_Size', 'Single_Item_Order_Ratio']].head())

   CustomerID  Basket_Diversity_Index  Avg_Basket_Size  \
0       12346                0.000000         0.000000   
1       12347                0.076716       351.142857   
2       12348                0.013191       585.250000   
3       12349                0.115689       631.000000   
4       12350                0.086294       197.000000   

   Single_Item_Order_Ratio  
0                      1.0  
1                      0.0  
2                      0.0  
3                      0.0  
4                      0.0  


In [120]:
rfm_features

,index,CustomerID,Recency,Active_Months,Purchase_Span,Frequency,Monetary,AOV,Avg_Items_per_Order,Hour_Entropy,...,Weekend_Ratio,IPI_Mean,IPI_Std,CV_Interval,Basket_Diversity_Index,Avg_Basket_Size,Avg_Unique_Items_per_Basket,Large_Basket_Ratio,Single_Item_Order_Ratio,Max_Single_Order_Revenue
0,0,12346,325,1,0,2,0.00,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.00,0.5,1.0,77183.60
1,1,12347,2,7,365,7,4310.00,615.714286,0.0,1.636439,...,0.000000,60.333333,18.478817,0.306279,0.076716,351.142857,26.00,1.0,0.0,1294.32
2,2,12348,75,4,282,4,1797.24,449.310000,0.0,0.923106,...,0.096774,94.000000,70.149840,0.746275,0.013191,585.250000,6.75,1.0,0.0,892.80
3,3,12349,18,1,0,1,1757.55,1757.550000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.115689,631.000000,73.00,1.0,0.0,1757.55
4,4,12350,310,1,0,1,334.40,334.400000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.086294,197.000000,17.00,1.0,0.0,334.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6726,6726,1481435,1,1,0,1,3.35,3.350000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000,2.00,0.0,0.0,3.35
6727,6727,1481439,1,1,0,1,6637.59,6637.590000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.363272,1748.000000,635.00,1.0,0.0,6637.59
6728,6728,1481492,0,1,0,1,7689.23,7689.230000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.363501,2011.000000,731.00,1.0,0.0,7689.23
6729,6729,1481497,0,1,0,1,3217.20,3217.200000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.085627,654.000000,56.00,1.0,0.0,3217.20


## 카테고리 4: 제품 충성도 변수 (유도성·타성 관련)
제품 재구매 패턴과 신규 카테고리 진입 비중을 통해 사용자의 브랜드 고착도를 측정함.

| 변수명 | 계산 방법 | 넛지 연결 |
| :--- | :--- | :--- |
| **Repurchase Rate** | 2회 이상 구매한 StockCode 수 / 전체 구매 품목 수 | 유도성·타성 측정 |
| **Top Product Concentration** | 가장 많이 구매한 제품의 구매 비율 | 브랜드 고착 정도 파악 |
| **Unique Products Ratio** | 고유 StockCode 수 / 총 구매 건수 | 탐색 성향 측정 |
| **Avg Repurchase Interval** | 동일 제품 재구매 간격 평균(일) | 유도성 반응 속도 측정 |
| **New Category Ratio** | 시계열상 신규 카테고리 첫 구매 비율 | 흥미성·탐색 성향 측정 |

In [127]:
# 1. 기초 데이터 준비: 카테고리 대용으로 Description의 첫 단어 추출 
df_final['Category'] = df_final['Description'].str.split().str[0]

# 2. 고객별 제품 구매 통계 계산 
product_stats = df_final.groupby('CustomerID').agg({
    'StockCode': ['count', 'nunique'],
    'Category': 'nunique'
})
product_stats.columns = ['Total_Purchases', 'Unique_Stocks', 'Unique_Categories']

# 3. 상세 지표 계산 
# (1) Repurchase Rate: 2회 이상 구매한 품목 수 / 전체 고유 품목 수
def calc_repurchase_count(x):
    counts = x.value_counts()
    return (counts >= 2).sum()

repurchase_counts = df_final.groupby('CustomerID')['StockCode'].apply(calc_repurchase_count)
product_stats['Repurchase_Rate'] = repurchase_counts / product_stats['Unique_Stocks']

# (2) Top Product Concentration: 최다 구매 품목 건수 / 총 구매 건수
def calc_top_product_ratio(x):
    if len(x) == 0: return 0
    return x.value_counts().max() / len(x)

product_stats['Top_Product_Concentration'] = df_final.groupby('CustomerID')['StockCode'].apply(calc_top_product_ratio)

# (3) Unique Products Ratio: 고유 품목 수 / 총 구매 건수
product_stats['Unique_Products_Ratio'] = product_stats['Unique_Stocks'] / product_stats['Total_Purchases']

# 4. Avg Repurchase Interval: 동일 제품 재구매 간격 평균 
df_sorted = df_final.sort_values(['CustomerID', 'StockCode', 'InvoiceDate'])
df_sorted['prev_date'] = df_sorted.groupby(['CustomerID', 'StockCode'])['InvoiceDate'].shift(1)
df_sorted['repurchase_interval'] = (df_sorted['InvoiceDate'] - df_sorted['prev_date']).dt.days
avg_interval = df_sorted.groupby('CustomerID')['repurchase_interval'].mean()

# 5. New Category Ratio: 시계열상 신규 카테고리 첫 구매 비율 
# 오류 수정: duplicated(subset=...)를 사용하여 고객별 중복 카테고리 판별
df_sorted_time = df_final.sort_values(['CustomerID', 'InvoiceDate'])
# subset에 CustomerID와 Category를 함께 넣어 '이 고객이 이 카테고리를 처음 샀는지' 판별합니다.
df_sorted_time['is_new_cat'] = ~df_sorted_time.duplicated(subset=['CustomerID', 'Category'])
new_cat_ratio = df_sorted_time.groupby('CustomerID')['is_new_cat'].mean()

# 6. 최종 병합 및 결측치 처리 [cite: 72]
category_4_features = pd.DataFrame({
    'Repurchase_Rate': product_stats['Repurchase_Rate'],
    'Top_Product_Concentration': product_stats['Top_Product_Concentration'],
    'Unique_Products_Ratio': product_stats['Unique_Products_Ratio'],
    'Avg_Repurchase_Interval': avg_interval,
    'New_Category_Ratio': new_cat_ratio
}).reset_index()

# 기존 rfm_features에 병합
rfm_features = pd.merge(rfm_features, category_4_features, on='CustomerID', how='left')
rfm_features = rfm_features.fillna(0)

print("카테고리 4 변수 생성 완료!")
print(rfm_features[['CustomerID', 'Repurchase_Rate', 'New_Category_Ratio']].head())

카테고리 4 변수 생성 완료!
   CustomerID  Repurchase_Rate  New_Category_Ratio
0       12346         1.000000            0.500000
1       12347         0.417476            0.307692
2       12348         0.318182            0.290323
3       12349         0.000000            0.616438
4       12350         0.000000            0.705882


In [129]:
rfm_features

,index,CustomerID,Recency,Active_Months,Purchase_Span,Frequency,Monetary,AOV,Avg_Items_per_Order,Hour_Entropy,...,Avg_Basket_Size,Avg_Unique_Items_per_Basket,Large_Basket_Ratio,Single_Item_Order_Ratio,Max_Single_Order_Revenue,Repurchase_Rate,Top_Product_Concentration,Unique_Products_Ratio,Avg_Repurchase_Interval,New_Category_Ratio
0,0,12346,325,1,0,2,0.00,0.000000,0.0,0.000000,...,0.000000,1.00,0.5,1.0,77183.60,1.000000,1.000000,0.500000,0.000000,0.500000
1,1,12347,2,7,365,7,4310.00,615.714286,0.0,1.636439,...,351.142857,26.00,1.0,0.0,1294.32,0.417476,0.032967,0.565934,110.544304,0.307692
2,2,12348,75,4,282,4,1797.24,449.310000,0.0,0.923106,...,585.250000,6.75,1.0,0.0,892.80,0.318182,0.129032,0.709677,69.777778,0.290323
3,3,12349,18,1,0,1,1757.55,1757.550000,0.0,0.000000,...,631.000000,73.00,1.0,0.0,1757.55,0.000000,0.013699,1.000000,0.000000,0.616438
4,4,12350,310,1,0,1,334.40,334.400000,0.0,0.000000,...,197.000000,17.00,1.0,0.0,334.40,0.000000,0.058824,1.000000,0.000000,0.705882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6726,6726,1481435,1,1,0,1,3.35,3.350000,0.0,0.000000,...,2.000000,2.00,0.0,0.0,3.35,0.000000,0.500000,1.000000,0.000000,1.000000
6727,6727,1481439,1,1,0,1,6637.59,6637.590000,0.0,0.000000,...,1748.000000,635.00,1.0,0.0,6637.59,0.000000,0.001575,1.000000,0.000000,0.388976
6728,6728,1481492,0,1,0,1,7689.23,7689.230000,0.0,0.000000,...,2011.000000,731.00,1.0,0.0,7689.23,0.000000,0.001368,1.000000,0.000000,0.393981
6729,6729,1481497,0,1,0,1,3217.20,3217.200000,0.0,0.000000,...,654.000000,56.00,1.0,0.0,3217.20,0.053571,0.033898,0.949153,0.000000,0.508475


## 카테고리 5: 가격 민감도 변수 (긍정성 관련)
제품 단가 선호도와 지출 트렌드를 통해 사용자의 가격 민감도 및 구매 성장 방향을 탐지함.

| 변수명 | 계산 방법 | 넛지 연결 |
| :--- | :--- | :--- |
| **Avg Unit Price** | 구매한 제품 단가 평균 | 가격대 선호 파악 |
| **Price Variance** | 구매 단가 표준편차 | 가격 일관성 측정 |
| **High Price Item Ratio** | 단가 상위 25% 제품 구매 비율 | 탐색적 vs 습관적 구매 구분 |
| **Low Price Item Ratio** | 단가 하위 25% 제품 구매 비율 | 저관여 습관 구매 측정 |
| **Revenue per Visit** | 방문당 Revenue 평균 | 지출 강도 측정 |
| **Revenue Trend** | 후반부 6개월 vs 전반부 6개월 Revenue 비율 | 구매 성장·이탈 방향 탐지 |

In [132]:
# 1. 단가 기준치 설정 (전체 상품 기준 상위 25%, 하위 25%)
q3_price = df_final['UnitPrice'].quantile(0.75)
q1_price = df_final['UnitPrice'].quantile(0.25)

# 2. 고객별 기본 가격 지표 집계
price_features = df_final.groupby('CustomerID').agg({
    'UnitPrice': ['mean', 'std'],
    'Revenue': 'sum'  # 전체 매출 (트렌드 계산용)
})
price_features.columns = ['Avg_Unit_Price', 'Price_Variance', 'Total_Revenue']

# 3. 고단가/저단가 상품 구매 비율 계산
df_final['is_high_price'] = (df_final['UnitPrice'] >= q3_price).astype(int)
df_final['is_low_price'] = (df_final['UnitPrice'] <= q1_price).astype(int)

high_low_ratio = df_final.groupby('CustomerID').agg({
    'is_high_price': 'mean',
    'is_low_price': 'mean'
}).rename(columns={'is_high_price': 'High_Price_Item_Ratio', 'is_low_price': 'Low_Price_Item_Ratio'})

# 4. Revenue per Visit (방문당 평균 매출)
# Frequency(주문 횟수)는 이미 rfm_features에 있으므로 나중에 계산함

# 5. Revenue Trend (후반 6개월 vs 전반 6개월) [cite: 59, 60]
# 기준일(2011-12-10)로부터 6개월 전인 2011-06-10을 분기점으로 설정
split_date = pd.to_datetime('2011-06-10')
first_half = df_final[df_final['InvoiceDate'] < split_date].groupby('CustomerID')['Revenue'].sum()
second_half = df_final[df_final['InvoiceDate'] >= split_date].groupby('CustomerID')['Revenue'].sum()

revenue_trend = (second_half / first_half).replace([np.inf, -np.inf], np.nan).fillna(0)
revenue_trend.name = 'Revenue_Trend'

# 6. 최종 병합
category_5_features = pd.concat([price_features, high_low_ratio, revenue_trend], axis=1).reset_index()

# 기존 rfm_features와 결합
rfm_features = pd.merge(rfm_features, category_5_features, on='CustomerID', how='left')

# Revenue per Visit 추가 계산
rfm_features['Revenue_per_Visit'] = rfm_features['Total_Revenue'] / rfm_features['Frequency']

# 중복된 Total_Revenue 컬럼 제거 및 결측치 처리
if 'Total_Revenue' in rfm_features.columns:
    rfm_features = rfm_features.drop(columns=['Total_Revenue'])
rfm_features = rfm_features.fillna(0)

print("카테고리 5 변수 생성 완료!")
print(rfm_features[['CustomerID', 'Avg_Unit_Price', 'High_Price_Item_Ratio', 'Revenue_Trend']].head())

카테고리 5 변수 생성 완료!
   CustomerID  Avg_Unit_Price  High_Price_Item_Ratio  Revenue_Trend
0       12346        1.040000               0.000000       0.000000
1       12347        2.644011               0.192308       0.953807
2       12348        5.764839               0.129032       0.208440
3       12349        8.289041               0.356164       0.000000
4       12350        3.841176               0.058824       0.000000


In [134]:
rfm_features

,index,CustomerID,Recency,Active_Months,Purchase_Span,Frequency,Monetary,AOV,Avg_Items_per_Order,Hour_Entropy,...,Top_Product_Concentration,Unique_Products_Ratio,Avg_Repurchase_Interval,New_Category_Ratio,Avg_Unit_Price,Price_Variance,High_Price_Item_Ratio,Low_Price_Item_Ratio,Revenue_Trend,Revenue_per_Visit
0,0,12346,325,1,0,2,0.00,0.000000,0.0,0.000000,...,1.000000,0.500000,0.000000,0.500000,1.040000,0.000000,0.000000,1.000000,0.000000,0.000000
1,1,12347,2,7,365,7,4310.00,615.714286,0.0,1.636439,...,0.032967,0.565934,110.544304,0.307692,2.644011,2.255381,0.192308,0.357143,0.953807,615.714286
2,2,12348,75,4,282,4,1797.24,449.310000,0.0,0.923106,...,0.129032,0.709677,69.777778,0.290323,5.764839,13.400323,0.129032,0.806452,0.208440,449.310000
3,3,12349,18,1,0,1,1757.55,1757.550000,0.0,0.000000,...,0.013699,1.000000,0.000000,0.616438,8.289041,35.028021,0.356164,0.191781,0.000000,1757.550000
4,4,12350,310,1,0,1,334.40,334.400000,0.0,0.000000,...,0.058824,1.000000,0.000000,0.705882,3.841176,9.334751,0.058824,0.411765,0.000000,334.400000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6726,6726,1481435,1,1,0,1,3.35,3.350000,0.0,0.000000,...,0.500000,1.000000,0.000000,1.000000,1.675000,0.601041,0.000000,0.500000,0.000000,3.350000
6727,6727,1481439,1,1,0,1,6637.59,6637.590000,0.0,0.000000,...,0.001575,1.000000,0.000000,0.388976,5.792236,37.282529,0.429921,0.155906,0.000000,6637.590000
6728,6728,1481492,0,1,0,1,7689.23,7689.230000,0.0,0.000000,...,0.001368,1.000000,0.000000,0.393981,5.446758,34.586468,0.403557,0.183311,0.000000,7689.230000
6729,6729,1481497,0,1,0,1,3217.20,3217.200000,0.0,0.000000,...,0.033898,0.949153,0.000000,0.508475,6.269661,3.925663,0.677966,0.000000,0.000000,3217.200000


## 카테고리 6: 반품·취소 행동 변수 (긍정성 관련)
반품 및 취소 패턴을 통해 고객의 구매 결정 불안정성과 서비스에 대한 신뢰 회복 속도를 측정함.

| 변수명 | 계산 방법 | 넛지 연결 |
| :--- | :--- | :--- |
| **Return Rate** | 반품 건수(Quantity < 0) / 전체 주문 수 | 긍정성·신뢰도 측정 |
| **Has Return** | 반품 경험 여부 (0/1) | 이진 지표 |
| **Days to Return Purchase** | 반품 후 다음 정상 구매까지 일수 | 긍정성 회복 속도 측정 |
| **Cancellation Rate** | C로 시작하는 InvoiceNo 비율 | 구매 결정 불안정성 측정 |
| **Return Recovery Ratio** | 반품 후 재진입 여부 (0/1) | 긍정성 넛지 효과 측정 |

In [139]:
# 1. 반품 및 정상 주문 분리 (InvoiceNo 'C' 시작 여부 및 Quantity 기준) [cite: 13, 61]
# 반품 주문: Quantity < 0
df_returns = df_final[df_final['Quantity'] < 0].copy()
# 정상 주문: Quantity > 0
df_normal = df_final[df_final['Quantity'] > 0].copy()

# 2. 고객별 기본 반품 통계 계산
# 전체 주문 수 계산 (CustomerID별 유니크 InvoiceNo) [cite: 45]
total_orders = df_final.groupby('CustomerID')['InvoiceNo'].nunique()
return_orders = df_returns.groupby('CustomerID')['InvoiceNo'].nunique()

category_6_features = pd.DataFrame(index=total_orders.index)
category_6_features['Return_Rate'] = (return_orders / total_orders).fillna(0)
category_6_features['Has_Return'] = (category_6_features['Return_Rate'] > 0).astype(int)

# 3. Cancellation Rate: C로 시작하는 InvoiceNo 비율 
c_orders = df_final[df_final['InvoiceNo'].astype(str).str.startswith('C')].groupby('CustomerID')['InvoiceNo'].nunique()
category_6_features['Cancellation_Rate'] = (c_orders / total_orders).fillna(0)

# 4. Days to Return Purchase & Return Recovery Ratio 계산 
# 각 반품 건에 대해 '그 이후' 가장 빠른 정상 주문 날짜 찾기
def calc_recovery_metrics(customer_id):
    cust_returns = df_returns[df_returns['CustomerID'] == customer_id]['InvoiceDate'].unique()
    cust_normals = df_normal[df_normal['CustomerID'] == customer_id]['InvoiceDate'].unique()
    
    if len(cust_returns) == 0:
        return pd.Series([np.nan, 0], index=['Days_to_Return_Purchase', 'Return_Recovery_Ratio'])
    
    recovery_days = []
    recovered_count = 0
    
    for r_date in cust_returns:
        # 반품일 이후의 정상 구매 건 필터링
        after_normal = cust_normals[cust_normals > r_date]
        if len(after_normal) > 0:
            recovery_days.append((after_normal.min() - r_date).days)
            recovered_count = 1 # 한 번이라도 복귀했다면 1로 설정
            
    avg_days = np.mean(recovery_days) if recovery_days else np.nan
    return pd.Series([avg_days, recovered_count], index=['Days_to_Return_Purchase', 'Return_Recovery_Ratio'])

# 전체 고객을 대상으로 회복 지표 산출 (시간이 다소 소요될 수 있음)
recovery_metrics = category_6_features.index.to_series().apply(calc_recovery_metrics)
category_6_features = pd.concat([category_6_features, recovery_metrics], axis=1)

# 5. 기존 rfm_features에 병합
rfm_features = pd.merge(rfm_features, category_6_features.reset_index(), on='CustomerID', how='left')
rfm_features = rfm_features.fillna(0)

print("카테고리 6 변수 생성 완료!")
print(rfm_features[['CustomerID', 'Return_Rate', 'Days_to_Return_Purchase', 'Return_Recovery_Ratio']].head())

카테고리 6 변수 생성 완료!
   CustomerID  Return_Rate  Days_to_Return_Purchase  Return_Recovery_Ratio
0       12346          0.5                      0.0                    0.0
1       12347          0.0                      0.0                    0.0
2       12348          0.0                      0.0                    0.0
3       12349          0.0                      0.0                    0.0
4       12350          0.0                      0.0                    0.0


## 카테고리 7: 사회적 비교 변수 (비교성 관련)
거주 국가의 트렌드와 개인의 소비 행태를 비교하여 사회적 증거에 대한 민감도와 집단 내 위치를 파악함.

| 변수명 | 계산 방법 | 넛지 연결 |
| :--- | :--- | :--- |
| **Country Trend Alignment** | 거주 국가 인기 품목과 개인 장바구니 일치율 | 비교성 측정 |
| **Country Purchase Rank** | 동일 국가 내 고객 Revenue 백분위 | 사회적 위치 파악 |
| **UK Flag** | UK 여부 (0/1) | 시장 세분화 기준 변수 |

In [142]:
# 1. 국가별 인기 품목(Top 10%) 추출
def get_country_trends(df):
    country_trends = {}
    for country in df['Country'].unique():
        top_items = df[df['Country'] == country]['StockCode'].value_counts()
        # 해당 국가 판매 품목 중 상위 10%를 인기 품목으로 정의
        threshold = max(1, int(len(top_items) * 0.1))
        country_trends[country] = set(top_items.head(threshold).index)
    return country_trends

country_top_items = get_country_trends(df_final)

# 2. Country Trend Alignment 계산 함수
def calc_alignment(row):
    customer_country = row['Country']
    customer_items = set(df_final[df_final['CustomerID'] == row.name]['StockCode'].unique())
    trend_items = country_top_items.get(customer_country, set())
    
    if not trend_items:
        return 0
    # 개인 구매 품목 중 국가 트렌드 아이템이 포함된 비율
    intersection = customer_items.intersection(trend_items)
    return len(intersection) / len(customer_items) if customer_items else 0

# 3. 고객별 국가 정보 및 UK Flag 생성
# rfm_features에 이미 CustomerID가 있으므로 이를 기준으로 국가 매핑
customer_country_map = df_final.groupby('CustomerID')['Country'].first()

category_7_features = pd.DataFrame(index=customer_country_map.index)
category_7_features['Country'] = customer_country_map
category_7_features['UK_Flag'] = (category_7_features['Country'] == 'United Kingdom').astype(int)

# 4. Country Trend Alignment 적용
category_7_features['Country_Trend_Alignment'] = category_7_features.apply(calc_alignment, axis=1)

# 5. Country Purchase Rank (국가 내 매출 백분위) 계산
# rfm_features의 Monetary(총 매출) 데이터를 활용
temp_df = pd.merge(category_7_features[['Country']], rfm_features[['CustomerID', 'Monetary']], on='CustomerID')

def get_percentile(group):
    return group.rank(pct=True)

category_7_features['Country_Purchase_Rank'] = temp_df.groupby('Country')['Monetary'].transform(get_percentile).values

# 6. 기존 rfm_features에 병합
rfm_features = pd.merge(rfm_features, category_7_features.drop(columns=['Country']).reset_index(), on='CustomerID', how='left')
rfm_features = rfm_features.fillna(0)

print("카테고리 7 변수 생성 완료!")
print(rfm_features[['CustomerID', 'Country_Trend_Alignment', 'Country_Purchase_Rank', 'UK_Flag']].head())

카테고리 7 변수 생성 완료!
   CustomerID  Country_Trend_Alignment  Country_Purchase_Rank  UK_Flag
0       12346                 0.000000               0.093329        1
1       12347                 0.097087               1.000000        0
2       12348                 0.227273               0.750000        0
3       12349                 0.164384               0.733333        0
4       12350                 0.235294               0.200000        0


### 카테고리 8: 일관성 변수 (일관성 넛지 관련)
구매 품목의 카테고리 변화와 지출 규모, 구매 리듬의 변동성을 분석하여 사용자의 행동 일관성을 측정함.

| 변수명 | 계산 방법 | 넛지 연결 |
| :--- | :--- | :--- |
| **Category Concentration Std** | 시계열상 카테고리 점유율 변화의 표준편차 | 낮을수록 일관된 취향으로 분류 |
| **Revenue Consistency** | 주문별 Revenue의 변동계수(CV) | 지출 일관성 측정 |
| **Purchase Rhythm Score** | 구매 간격 CV의 역수(1/CV) | 구매 리듬 안정성 측정 |

In [146]:
# 1. Category Concentration Std: 시간 흐름에 따른 카테고리 점유율의 변동성 측정
# 데이터를 월별로 나누어 각 월별 카테고리 비중의 표준편차를 계산함
df_final['month_year'] = df_final['InvoiceDate'].dt.to_period('M')
category_monthly = df_final.groupby(['CustomerID', 'month_year', 'Category']).size().unstack(fill_value=0)
category_share = category_monthly.div(category_monthly.sum(axis=1), axis=0)

# 고객별로 각 카테고리 점유율의 표준편차를 구한 뒤 그 평균을 취함
category_consistency = category_share.groupby(level=0).std().mean(axis=1)
category_consistency.name = 'Category_Concentration_Std'

# 2. Revenue Consistency: 주문별 매출의 변동계수(CV = Std / Mean)
# 값이 낮을수록 매번 비슷한 금액을 지출함을 의미함
order_revenue = df_final.groupby(['CustomerID', 'InvoiceNo'])['Revenue'].sum().reset_index()
revenue_stats = order_revenue.groupby('CustomerID')['Revenue'].agg(['mean', 'std'])
revenue_cv = (revenue_stats['std'] / revenue_stats['mean']).fillna(0)
revenue_cv.name = 'Revenue_Consistency'

# 3. Purchase Rhythm Score: 구매 간격 CV의 역수 (1 / (Std/Mean)) [cite: 65]
# 카테고리 2에서 계산했던 CV_Interval 활용 (값이 높을수록 리듬이 일정함)
# rfm_features에 이미 CV_Interval이 계산되어 있다고 가정함
purchase_rhythm = (1 / rfm_features.set_index('CustomerID')['CV_Interval']).replace([np.inf, -np.inf], 0).fillna(0)
purchase_rhythm.name = 'Purchase_Rhythm_Score'

# 4. 최종 병합
category_8_features = pd.concat([category_consistency, revenue_cv, purchase_rhythm], axis=1).reset_index()

# 기존 rfm_features에 결합
rfm_features = pd.merge(rfm_features, category_8_features, on='CustomerID', how='left')
rfm_features = rfm_features.fillna(0)

print("카테고리 8 변수 생성 및 전체 변수 통합 완료!")
print(rfm_features[['CustomerID', 'Category_Concentration_Std', 'Revenue_Consistency', 'Purchase_Rhythm_Score']].head())

카테고리 8 변수 생성 및 전체 변수 통합 완료!
   CustomerID  Category_Concentration_Std  Revenue_Consistency  \
0       12346                    0.000000                  inf   
1       12347                    0.001635             0.553943   
2       12348                    0.001369             0.670272   
3       12349                    0.000000             0.000000   
4       12350                    0.000000             0.000000   

   Purchase_Rhythm_Score  
0               0.000000  
1               3.265000  
2               1.339989  
3               0.000000  
4               0.000000  


### 종속변수 라벨링 계획
선행연구와의 차별성을 위해 단일 지표가 아닌 복수의 종속변수 후보군을 설정하여 다차원적으로 분석함.

| 종속변수 | 라벨링 기준 | 분석 및 라벨링 방향 |
| :--- | :--- | :--- |
| **이탈 여부 (Main)** | Recency > 90일 $\rightarrow$ 이탈(1) | 기준일(2011-12-10) 기준 비활성 기간으로 라벨링 |
| **충동 구매 지수** | (AOV 1.5배 초과) & (신규 카테고리 포함) | 평소 패턴 대비 단가, 수량, 카테고리의 변동 폭 수치화 |
| **구매 맥락** | Hour & DayOfWeek Entropy 하위 33% | 루틴(0)과 탐색(1)을 구분하여 분석 |
| **예산 준수도** | 과거 평균 AOV 대비 주문 총액 초과 여부 | 지출 통제력 및 예산 민감도 측정 |

In [153]:
import pandas as pd
import numpy as np

# 1. 누락된 기초 컬럼 재생성 (is_new_cat)
# 시계열 순으로 정렬 후, 고객별로 처음 등장하는 카테고리인지를 판별합니다. [cite: 54]
df_sorted_time = df_final.sort_values(['CustomerID', 'InvoiceDate'])
df_final['is_new_cat'] = ~df_sorted_time.duplicated(subset=['CustomerID', 'Category'])
df_final['is_new_cat'] = df_final['is_new_cat'].astype(int)

# 2. 이탈 여부 (Main): Recency 기준 90일 초과 시 1 [cite: 67]
rfm_features['Target_Churn'] = (rfm_features['Recency'] > 90).astype(int)

# 3. 구매 맥락: 엔트로피 하위 33% 기준 루틴(0) vs 탐색(1) [cite: 67]
hour_threshold = rfm_features['Hour_Entropy'].quantile(0.33)
day_threshold = rfm_features['DayOfWeek_Entropy'].quantile(0.33)

rfm_features['Target_Context'] = 1  # 기본 탐색(1) 설정
rfm_features.loc[
    (rfm_features['Hour_Entropy'] <= hour_threshold) & 
    (rfm_features['DayOfWeek_Entropy'] <= day_threshold), 
    'Target_Context'
] = 0

# 4. 충동 구매 지수 및 예산 준수도 (주문 단위 분석) [cite: 67, 70]
order_labels = df_final.groupby(['CustomerID', 'InvoiceNo']).agg({
    'Revenue': 'sum',
    'is_new_cat': 'max'  # 해당 주문에 신규 카테고리 포함 여부
}).reset_index()

# 고객별 평균 AOV 매핑
order_labels = pd.merge(order_labels, rfm_features[['CustomerID', 'AOV']], on='CustomerID')

# 예산 준수도: 주문 총액이 과거 평균(AOV)을 초과하면 1 [cite: 70]
order_labels['Target_Budget'] = (order_labels['Revenue'] > order_labels['AOV']).astype(int)

# 충동 구매: (수익 > AOV * 1.5) AND (신규 카테고리 포함) [cite: 67]
order_labels['Target_Impulse'] = (
    (order_labels['Revenue'] > order_labels['AOV'] * 1.5) & 
    (order_labels['is_new_cat'] == 1)
).astype(int)

# 5. 고객 단위로 최종 라벨링 통합
# 각 고객이 한 번이라도 충동 구매나 예산 초과를 했다면 1로 정의합니다.
final_labels = order_labels.groupby('CustomerID').agg({
    'Target_Budget': 'max',
    'Target_Impulse': 'max'
}).reset_index()

# 6. rfm_features와 최종 결합
rfm_features = pd.merge(rfm_features, final_labels, on='CustomerID', how='left')
rfm_features[['Target_Budget', 'Target_Impulse']] = rfm_features[['Target_Budget', 'Target_Impulse']].fillna(0)

print("종속변수 라벨링이 성공적으로 완료되었습니다.")
print(rfm_features[['CustomerID', 'Target_Churn', 'Target_Context', 'Target_Impulse', 'Target_Budget']].head())

종속변수 라벨링이 성공적으로 완료되었습니다.
   CustomerID  Target_Churn  Target_Context  Target_Impulse  Target_Budget
0       12346             1               0               1              1
1       12347             0               1               1              1
2       12348             0               1               1              1
3       12349             0               0               0              0
4       12350             1               0               0              0


In [155]:
rfm_features

,index,CustomerID,Recency,Active_Months,Purchase_Span,Frequency,Monetary,AOV,Avg_Items_per_Order,Hour_Entropy,...,UK_Flag,Country_Trend_Alignment,Country_Purchase_Rank,Category_Concentration_Std,Revenue_Consistency,Purchase_Rhythm_Score,Target_Churn,Target_Context,Target_Budget,Target_Impulse
0,0,12346,325,1,0,2,0.00,0.000000,0.0,0.000000,...,1,0.000000,0.093329,0.000000,inf,0.000000,1,0,1,1
1,1,12347,2,7,365,7,4310.00,615.714286,0.0,1.636439,...,0,0.097087,1.000000,0.001635,0.553943,3.265000,0,1,1,1
2,2,12348,75,4,282,4,1797.24,449.310000,0.0,0.923106,...,0,0.227273,0.750000,0.001369,0.670272,1.339989,0,1,1,1
3,3,12349,18,1,0,1,1757.55,1757.550000,0.0,0.000000,...,0,0.164384,0.733333,0.000000,0.000000,0.000000,0,0,0,0
4,4,12350,310,1,0,1,334.40,334.400000,0.0,0.000000,...,0,0.235294,0.200000,0.000000,0.000000,0.000000,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6726,6726,1481435,1,1,0,1,3.35,3.350000,0.0,0.000000,...,1,0.000000,0.166693,0.000000,0.000000,0.000000,0,0,0,0
6727,6727,1481439,1,1,0,1,6637.59,6637.590000,0.0,0.000000,...,1,0.333858,0.971777,0.000000,0.000000,0.000000,0,0,0,0
6728,6728,1481492,0,1,0,1,7689.23,7689.230000,0.0,0.000000,...,1,0.279070,0.979474,0.000000,0.000000,0.000000,0,0,0,0
6729,6729,1481497,0,1,0,1,3217.20,3217.200000,0.0,0.000000,...,1,0.732143,0.914047,0.000000,0.000000,0.000000,0,0,0,0


# rfm_features 분포 확인

In [160]:
# 1. 기초 통계량 요약 (Transpose하여 보기 편하게 출력)
print("### [1] 수치형 변수 통계 요약")
display(rfm_features.describe().T)

# 2. 고유값(Unique Values) 개수 확인
# 값이 1개인 변수는 모델 학습에 아무런 정보가 되지 않으므로 삭제 대상입니다.
print("\n### [2] 변수별 고유값 개수 확인")
nunique_counts = rfm_features.nunique()
display(nunique_counts)

# 고유값이 1개인 변수 추출
single_value_cols = nunique_counts[nunique_counts <= 1].index.tolist()
if single_value_cols:
    print(f"⚠️ 경고: 다음 변수들은 값이 1개뿐입니다: {single_value_cols}")
else:
    print("✅ 모든 변수가 2개 이상의 고유값을 가지고 있습니다.")

### [1] 수치형 변수 통계 요약


,count,mean,std,min,25%,50%,75%,max
index,6731.0,3.365000e+03,1943.216663,0.000000,1682.500000,3365.000000,5.047500e+03,6.730000e+03
CustomerID,6731.0,5.205769e+05,687962.348077,12346.000000,14621.500000,16900.000000,1.447684e+06,1.481498e+06
Recency,6731.0,1.234647e+02,113.184276,0.000000,24.000000,79.000000,2.130000e+02,3.730000e+02
Active_Months,6731.0,2.382113e+00,2.402550,1.000000,1.000000,1.000000,3.000000e+00,1.300000e+01
Purchase_Span,6731.0,8.663839e+01,124.549976,0.000000,0.000000,0.000000,1.800000e+02,3.730000e+02
Frequency,6731.0,3.647155e+00,7.773302,1.000000,1.000000,1.000000,4.000000e+00,2.480000e+02
Monetary,6731.0,1.448187e+03,6809.546317,-17836.460000,98.005000,457.810000,1.384990e+03,2.794890e+05
AOV,6731.0,4.202532e+02,1490.545798,-17836.460000,71.953333,205.970000,4.010055e+02,5.294094e+04
Avg_Items_per_Order,6731.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
Hour_Entropy,6731.0,4.314801e-01,0.582976,0.000000,0.000000,0.000000,8.392155e-01,2.317694e+00



### [2] 변수별 고유값 개수 확인


index                          6731
CustomerID                     6731
Recency                         304
Active_Months                    13
Purchase_Span                   374
Frequency                        65
Monetary                       5732
AOV                            5741
Avg_Items_per_Order               1
Hour_Entropy                   2289
Preferred_Hour                   14
DayOfWeek_Entropy              2142
Month_Entropy                  2221
Weekend_Ratio                   855
IPI_Mean                       1347
IPI_Std                        1927
CV_Interval                    2101
Basket_Diversity_Index         4843
Avg_Basket_Size                2548
Avg_Unique_Items_per_Basket    1321
Large_Basket_Ratio              157
Single_Item_Order_Ratio         155
Max_Single_Order_Revenue       5651
Repurchase_Rate                1463
Top_Product_Concentration      1129
Unique_Products_Ratio          1681
Avg_Repurchase_Interval        2082
New_Category_Ratio          

⚠️ 경고: 다음 변수들은 값이 1개뿐입니다: ['Avg_Items_per_Order']


🛠️ Phase 1-2: 데이터 정제 및 변수 정상화 (Data Cleaning)

기초 파생변수 생성 후 기술 통계 분석을 통해 발견된 수치적 왜곡 및 오류를 해결하기 위해 다음과 같은 **데이터 필터링 및 변수 재건** 과정을 수행함.

1. 주요 수정 사항 및 해결 로직

| 대상 변수 | 발견된 문제점 | 해결 방법 (Logic) | 연구적 기대효과 |
| :--- | :--- | :--- | :--- |
| **Monetary / AOV** | 환불액 반영으로 인한 음수(-) 발생 | **환불(Quantity < 0) 제외**, 순수 구매 로그로 재산출 | 고객의 순수 구매력 및 객단가 선호도 정밀 측정 |
| **Avg Items per Order** | std=0, 전 구간 0 값 발생 | **순수 구매 수량** 기준 주문당 평균치 재계산 | 실제 장바구니 규모(Scale)에 대한 노이즈 제거 |
| **Revenue Consistency** | `inf`(무한대) 및 `NaN` 발생 | CV 산출 시 분모 0 처리 및 무한대 0 치환 | 지출 일관성 지표의 통계적 안정성 확보 |
| **Rhythm / Interval** | 시계열 부족으로 인한 계산 오류 | `replace([inf, -inf], 0)` 및 결측치 보정 | 불규칙 구매자와 루틴 구매자의 명확한 변별력 확보 |

In [162]:
# --- 1. Avg_Items_per_Order (재계산) ---
# 원인: 분모가 0이거나 데이터 누락 -> 주문당 평균 수량으로 다시 정의
# 총 수량(절대값) / 총 주문 횟수
total_qty_per_cust = df_final.groupby('CustomerID')['Quantity'].apply(lambda x: x.abs().sum())
rfm_features['Avg_Items_per_Order'] = total_qty_per_cust / rfm_features['Frequency']

# --- 2. Revenue_Consistency (무한대 해결) ---
# 원인: 평균이 0이거나 데이터가 1개일 때 발생 -> 안정적인 CV 계산
order_revs = df_final.groupby(['CustomerID', 'InvoiceNo'])['Revenue'].sum().abs()
rev_stats = order_revs.groupby('CustomerID').agg(['mean', 'std'])

# std가 NaN이거나 mean이 0인 경우를 대비해 fillna(0) 처리
rfm_features['Revenue_Consistency'] = (rev_stats['std'] / rev_stats['mean']).fillna(0).replace([np.inf, -np.inf], 0)

# --- 3. Monetary, AOV, UnitPrice (음수 보정) ---
# 원인: 반품액이 구매액보다 커서 발생 -> '구매 성향' 분석을 위해 절대값 또는 0 하한선 적용
money_cols = ['Monetary', 'AOV', 'Avg_Unit_Price', 'Max_Single_Order_Revenue', 'Revenue_per_Visit']

for col in money_cols:
    if col in rfm_features.columns:
        # 반품액을 차감한 결과가 음수라면, 해당 고객의 '구매 활동성'을 0으로 보거나 
        # 분석 목적에 따라 절대값을 취함 (여기서는 0 하한선 적용 후 매우 작은 값 처리)
        rfm_features[col] = rfm_features[col].clip(lower=0)

# --- 4. 추가 클리닝: Purchase_Rhythm_Score ---
# 1/CV 계산 시 분모가 0이 되어 inf가 된 것들 처리
rfm_features['Purchase_Rhythm_Score'] = rfm_features['Purchase_Rhythm_Score'].replace([np.inf, -np.inf], 0).fillna(0)

print("✅ 모든 문제 변수가 재건되었습니다. 다시 describe()를 돌려 확인해보세요!")

✅ 모든 문제 변수가 재건되었습니다. 다시 describe()를 돌려 확인해보세요!


In [165]:
# 1. 순수 구매 데이터만 필터링 (환불 제외)
df_purchases = df_final[df_final['Quantity'] > 0].copy()

# 2. 순수 구매 기준 Monetary 및 AOV 재계산
pure_monetary = df_purchases.groupby('CustomerID')['Revenue'].sum()
pure_frequency = df_purchases.groupby('CustomerID')['InvoiceNo'].nunique()

# 3. rfm_features 업데이트
rfm_features['Monetary'] = rfm_features['CustomerID'].map(pure_monetary).fillna(0)
rfm_features['Frequency_Pure'] = rfm_features['CustomerID'].map(pure_frequency).fillna(0)
rfm_features['AOV'] = rfm_features['Monetary'] / rfm_features['Frequency_Pure']

# 4. Avg Items per Order (순수 구매 수량 기준)
pure_total_qty = df_purchases.groupby('CustomerID')['Quantity'].sum()
rfm_features['Avg_Items_per_Order'] = pure_total_qty / rfm_features['Frequency_Pure']

# 5. Avg Unit Price (순수 구매 단가 기준)
pure_unit_price = df_purchases.groupby('CustomerID')['UnitPrice'].mean()
rfm_features['Avg_Unit_Price'] = rfm_features['CustomerID'].map(pure_unit_price).fillna(0)

# 무한대 및 결측치 최종 정리
rfm_features = rfm_features.replace([np.inf, -np.inf], 0).fillna(0)

print("✅ 환불 데이터를 제외한 '순수 구매 성향' 변수 재건 완료!")

✅ 환불 데이터를 제외한 '순수 구매 성향' 변수 재건 완료!


In [168]:
# 1. 기초 통계량 요약 (Transpose하여 보기 편하게 출력)
print("### [1] 수치형 변수 통계 요약")
display(rfm_features.describe().T)

# 2. 고유값(Unique Values) 개수 확인
# 값이 1개인 변수는 모델 학습에 아무런 정보가 되지 않으므로 삭제 대상입니다.
print("\n### [2] 변수별 고유값 개수 확인")
nunique_counts = rfm_features.nunique()
display(nunique_counts)

# 고유값이 1개인 변수 추출
single_value_cols = nunique_counts[nunique_counts <= 1].index.tolist()
if single_value_cols:
    print(f"⚠️ 경고: 다음 변수들은 값이 1개뿐입니다: {single_value_cols}")
else:
    print("✅ 모든 변수가 2개 이상의 고유값을 가지고 있습니다.")

### [1] 수치형 변수 통계 요약


,count,mean,std,min,25%,50%,75%,max
index,6731.0,3365.000000,1943.216663,0.000000,1682.500000,3365.000000,5.047500e+03,6.730000e+03
CustomerID,6731.0,520576.886941,687962.348077,12346.000000,14621.500000,16900.000000,1.447684e+06,1.481498e+06
Recency,6731.0,123.464715,113.184276,0.000000,24.000000,79.000000,2.130000e+02,3.730000e+02
Active_Months,6731.0,2.382113,2.402550,1.000000,1.000000,1.000000,3.000000e+00,1.300000e+01
Purchase_Span,6731.0,86.638390,124.549976,0.000000,0.000000,0.000000,1.800000e+02,3.730000e+02
Frequency,6731.0,3.647155,7.773302,1.000000,1.000000,1.000000,4.000000e+00,2.480000e+02
Monetary,6731.0,1581.423328,7361.122491,-11062.060000,102.810000,468.770000,1.417420e+03,2.802060e+05
AOV,6731.0,527.618555,1951.448936,-11062.060000,84.541667,243.760000,4.552750e+02,8.423625e+04
Avg_Items_per_Order,6731.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
Hour_Entropy,6731.0,0.431480,0.582976,0.000000,0.000000,0.000000,8.392155e-01,2.317694e+00



### [2] 변수별 고유값 개수 확인


index                          6731
CustomerID                     6731
Recency                         304
Active_Months                    13
Purchase_Span                   374
Frequency                        65
Monetary                       5522
AOV                            5534
Avg_Items_per_Order               1
Hour_Entropy                   2289
Preferred_Hour                   14
DayOfWeek_Entropy              2142
Month_Entropy                  2221
Weekend_Ratio                   855
IPI_Mean                       1347
IPI_Std                        1927
CV_Interval                    2101
Basket_Diversity_Index         4843
Avg_Basket_Size                2548
Avg_Unique_Items_per_Basket    1321
Large_Basket_Ratio              157
Single_Item_Order_Ratio         155
Max_Single_Order_Revenue       5446
Repurchase_Rate                1463
Top_Product_Concentration      1129
Unique_Products_Ratio          1681
Avg_Repurchase_Interval        2082
New_Category_Ratio          

⚠️ 경고: 다음 변수들은 값이 1개뿐입니다: ['Avg_Items_per_Order', 'Revenue_Consistency']


Phase 1-3: 최종 분석 데이터셋(Feature Set) 확정

통계 요약 분석을 통해 변별력이 없는 변수를 제거하고, 모델의 안정성을 해치는 극단적 이상치를 제어하여 최종 학습 데이터를 구성함.

1. 데이터 클리닝 및 변수 선택 전략
| 조치 사항 | 대상 변수 | 사유 |
| :--- | :--- | :--- |
| **변수 제거** | `Avg_Items_per_Order`, `Revenue_Consistency` | 모든 고객이 동일 값을 가짐 (Zero Variance) |
| **로그 변환 고려** | `Monetary`, `AOV`, `Purchase_Rhythm_Score` | 극심한 우편향 분포로 인한 모델 왜곡 방지 |
| **기준 지표 통합** | `Frequency_Pure` 사용 | 환불 데이터를 제외한 실제 구매 횟수 반영 |


In [173]:
# [Step 1] 변별력 없는 변수 및 식별자 제거
drop_cols = ['Avg_Items_per_Order', 'Revenue_Consistency', 'CustomerID', 'index', 'Frequency']
# Target(정답) 변수들도 Feature(X)에서는 제외
target_cols = ['Target_Churn', 'Target_Context', 'Target_Budget', 'Target_Impulse']

# 최종 Feature 데이터셋 X 생성
X = rfm_features.drop(columns=drop_cols + target_cols)

# [Step 2] 수치 안정화 (무한대 및 결측치 최종 체크)
X = X.replace([np.inf, -np.inf], 0).fillna(0)

# [Step 3] 이상치 영향 최소화를 위한 로그 변환 (선택 사항이나 권장)
# 금액 관련 변수 등 왜도가 심한 변수들에 log1p 적용
skewed_cols = ['Monetary', 'AOV', 'Max_Single_Order_Revenue', 'Revenue_per_Visit', 'Purchase_Rhythm_Score']
for col in skewed_cols:
    X[col] = np.log1p(X[col])

# 정답지 y 설정 (가장 중요한 Target_Churn 기준)
y = rfm_features['Target_Churn']

print(f"✅ 최종 학습 데이터 준비 완료!")
print(f"- 총 고객 수(Rows): {X.shape[0]}명")
print(f"- 사용된 변수 수(Features): {X.shape[1]}개")

✅ 최종 학습 데이터 준비 완료!
- 총 고객 수(Rows): 6731명
- 사용된 변수 수(Features): 41개


/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [175]:
# 최종 데이터프레임 저장
rfm_features.to_csv('final_customer_data_20260408.csv', index=False)
print("✅ 최종 데이터셋이 'final_customer_data_20260408.csv'로 저장되었습니다.")

✅ 최종 데이터셋이 'final_customer_data_20260408.csv'로 저장되었습니다.
